# Fold-hizali yeniden egitim — Tuna'nin folds.parquet'iyle

**Amac:** Tuna'nin "senin OOF'un KFold(10), benim fold'uma hizasiz -> rw-OOF iyimser-yanli" itirazini kapatmak.
v45 tarifimizi (LGBM+CatBoost+XGB+MLP, segment-yil TE + kohort-z + year-norm) **aynen** tutup,
fold kaynagini Tuna'nin RepeatedStratified 5x3 (`folds.parquet`) yapiyoruz.

**Cikti:** `ourteam_oof_tunafolds.npy` + `ourteam_test_tunafolds.npy` + `preds_tunafolds.npz`
-> bunlari Tuna'ya yolla, paired-gate'i kendi zemininde calistirin.

**Calistirma:** Runtime > Change runtime type > **A100 GPU**. Sonra hucreleri sirayla calistir.
Tahmini sure: ~15-25 dk (15 split x 2 seed x 4 model).

## 1) Paketi yukle
Asagidaki hucreyi calistir, **`colab_refit_pkg.zip`**'i sec (lokalde olusturuldu).

In [ ]:
from google.colab import files
up = files.upload()  # colab_refit_pkg.zip sec
import zipfile, os
zname = [k for k in up if k.endswith('.zip')][0]
with zipfile.ZipFile(zname) as z:
    z.extractall('.')
# zip icindeki klasoru duzlestir
if os.path.isdir('colab_refit_pkg'):
    for f in os.listdir('colab_refit_pkg'):
        os.replace(f'colab_refit_pkg/{f}', f)
print('dosyalar:', sorted(os.listdir('.'))[:30])

## 2) GPU + kutuphaneler
CatBoost/XGBoost GPU. transformers gerekmiyor (metin skalarlari hazir npy).

In [ ]:
!nvidia-smi -L
!pip -q install lightgbm catboost xgboost pyarrow 2>/dev/null
import torch  # sadece cuda kontrolu icin (varsa)
import lightgbm, catboost, xgboost, pandas as pd, numpy as np
print('lgbm', lightgbm.__version__, '| cat', catboost.__version__, '| xgb', xgboost.__version__, '| pandas', pd.__version__)

## 3) Sanity: folds.parquet hizali mi?

In [ ]:
import pandas as pd
fd = pd.read_parquet('folds.parquet')
ft = pd.read_parquet('feat_train.parquet')
print('folds:', fd.shape, '| kolonlar:', list(fd.columns))
print('repeat:', sorted(fd.repeat.unique()), '| fold:', sorted(fd.fold.unique()))
print('feat_train:', ft.shape)
print('ID ortusme:', len(set(fd.student_id) & set(ft.student_id)), '/ 10000')

## 4) TAM KOSU — Tuna fold'uyla yeniden egit
15 split x 2 seed x {LGBM, XGB(GPU), CatBoost(GPU), MLP}. Her split sonunda satir basar.

In [ ]:
import refit_tunafolds_core as R
from pathlib import Path
# Colab'da her sey cwd'de:
def find2(name):
    p = Path('.') / name
    if p.exists():
        return p
    raise FileNotFoundError(name)
R.find = find2
R.main()

## 5) Hizli paired-test (Colab'da on-kontrol)
Birlesme kazanci fold-hizalandiktan SONRA hayatta mi? (Tuna'nin asil itirazi).

In [ ]:
import numpy as np, pandas as pd
tr = pd.read_csv('train.csv', encoding='utf-8-sig')
te = pd.read_csv('test_x.csv', encoding='utf-8-sig')
y = tr['career_success_score'].values.astype(float)
yr = tr['application_year'].values
def yw(a, b, cap=(0.3, 2.5)):
    pa = pd.Series(a).value_counts(normalize=True); pb = pd.Series(b).value_counts(normalize=True)
    w = pd.Series(a).map(pb / pa).fillna(1.0).clip(*cap).values
    return w * len(w) / w.sum()
w = yw(tr['application_year'], te['application_year'])
def rw(p): return float(np.average((y - np.clip(p, 0, 100))**2, weights=w))

sub2 = np.load('sub2_oof.npy')
ours = np.load('ourteam_oof_tunafolds.npy')
print(f'sub2 (Tuna)          rw = {rw(sub2):.3f}')
print(f'bizim TUNA-FOLD      rw = {rw(ours):.3f}')
print(f'corr                 = {np.corrcoef(sub2, ours)[0,1]:.4f}')
best = None
for a in [0.40, 0.45, 0.50, 0.55, 0.60]:
    b = a*sub2 + (1-a)*ours; m = rw(b)
    print(f'  {a:.2f}*sub2 + {1-a:.2f}*bizim = {m:.3f}  ({m-rw(sub2):+.3f})')
    if best is None or m < best[1]: best = (a, m)
a, m = best
blend = a*sub2 + (1-a)*ours
# bootstrap paired-test
se_b = (y-sub2)**2; se_m = (y-blend)**2; delta = se_b - se_m
rng = np.random.default_rng(42); n=len(y); B=2000; boots=np.empty(B); idx=np.arange(n)
for i in range(B):
    s = rng.choice(idx, n, replace=True); boots[i] = np.average(delta[s], weights=w[s])
lo, hi = np.percentile(boots, [2.5, 97.5])
print(f'\nEn iyi a={a:.2f}, blend rw={m:.3f} ({m-rw(sub2):+.3f})')
print(f'Bootstrap %95 CI = [{lo:+.3f}, {hi:+.3f}]  P(kazanc<=0)={(boots<=0).mean():.4f}')
hit = sum(np.average(delta[yr==u], weights=w[yr==u]) > 0 for u in np.unique(yr))
print(f'Yil-paneli: {hit}/{len(np.unique(yr))} yilda birlesme daha iyi')

## 6) Ciktilari indir
Bu 3 dosyayi Tuna'ya yolla -> o kendi `probe_compare_solutions.py` ile paired-gate'i (>=12/15, p<0.01) calistirsin.

In [ ]:
from google.colab import files
for f in ['ourteam_oof_tunafolds.npy', 'ourteam_test_tunafolds.npy', 'preds_tunafolds.npz']:
    files.download(f)